## My predictions

**1. Which language will need the most tokens?**

I expect **Chinese to be the one with most tokens**. I expect **English to need the fewest tokens** and **Turkish in between**.

**2. What will the larger BPE vocabulary change?**

Fewer tokens for every language and more characters per token. I expect the **biggest improvement for Chinese** since 10,000 slots are enough to give most common characters their own token. English should improve too and Turkish should sit between them: frequent stems and suffixes become tokens, but the huge number of different inflected forms is still split.

**3. What kinds of units will BPE learn?**

- **English:** frequent whole words (`the`, `of`, `and`, ` in`), common suffixes/prefixes (`ing`, `tion`, `ed`, `re`). Note that a leading space will be attached to word tokens because of how byte-level BPE handles whitespace.
- **Turkish:** I dont know much about the language so im planning to experiment it as i go.
- **Chinese:** Same as Turkish.

**4. Which tokenizer will work best for language modeling?**

My guess is the **BPE-2k or BPE-10k** and I lean to **BPE-2k** on this small dataset : shorter sequences than character-level (so the Transformer sees more context per position and training is cheaper) but with a small embedding table and few rare tokens that appear too seldom to be learned well. The character-level model should have the longest sequences and must learn spelling from scratch, so I expect it to be worst or at least the slowest. I'm least sure about BPE-10k: it has the shortest sequences, but many of its tokens will be rare in such a small corpus.
    

### Part 1

In [20]:
import os, random
import pandas as pd
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

DATA_DIR = "/srv/data/lt2326-h26/a1"

OUT_DIR = "outputs_part1"
os.makedirs(OUT_DIR, exist_ok=True)

LANGS = ["en", "tr", "zh"]           # the three languages
SEED = 2326                          # same seed as the dataset metadata
random.seed(SEED)

## 1.1 Load the data
Each file has one sentence per line. We read **train** to build tokenizers and **valid** to analyse them

In [21]:
def read_lines(path):
    # Reading a text file, one sentence per line, skipping empty lines
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]

train = {lang: read_lines(f"{DATA_DIR}/train/{lang}.txt") for lang in LANGS}
valid = {lang: read_lines(f"{DATA_DIR}/valid/{lang}.txt") for lang in LANGS}

# check: sentence and character counts if it matches metadata.json
for lang in LANGS:
    print(lang,
          "| train sentences:", len(train[lang]), "chars:", sum(len(s) for s in train[lang]),
          "| valid sentences:", len(valid[lang]), "chars:", sum(len(s) for s in valid[lang]))

en | train sentences: 36004 chars: 2946215 | valid sentences: 4540 chars: 368269
tr | train sentences: 30379 chars: 2946140 | valid sentences: 3735 chars: 368383
zh | train sentences: 75962 chars: 2946177 | valid sentences: 9554 chars: 368266


## 1.2 Tokenizer 1: character level

The vocabulary is every distinct character seen in the **training** data only.
A character that first shows up in validation/test (for example: a rare Chinese character) is mapped to a special `<unk>` token instead of crashing.
I also count how often that happens, so we know how much information this loses.

In [22]:
class CharTokenizer:
    def __init__(self, sentences):
        # sorted set of all characters in the training text 
        #id 0 is reserved for <unk>
        chars = sorted(set("".join(sentences)))
        self.itos = ["<unk>"] + chars                        # id ---> token
        self.stoi = {c: i for i, c in enumerate(self.itos)}  # token ----> id

    def encode(self, text):
        # unseen characters get id 0 (<unk>)
        return [self.stoi.get(c, 0) for c in text]

    def encode_batch(self, sentences):
        return [self.encode(s) for s in sentences]

    def to_tokens(self, text):
        # human-readable tokens (for the examples section)
        return [c if c in self.stoi else "<unk>" for c in text]

    def vocab_size(self):
        return len(self.itos)

    def unk_id(self):
        return 0

# Building it from aLL training sentences of all three languages (one shared vocabulary)
all_train = [s for lang in LANGS for s in train[lang]]
char_tok = CharTokenizer(all_train)
print("Character vocabulary size:", char_tok.vocab_size())

Character vocabulary size: 9443


## 1.3 Tokenizers 2 and 3: BPE with a small and a large vocabulary

**Choices and why**
- **Library:** Hugging Face `tokenizers` 
- **Byte-level BPE:** the starting alphabet is the 256 possible bytes, not characters. This has two advantages: (a) *no unknown tokens ever*, any text can be encoded; (b) the base alphabet is only 256 symbols. Chinese alone has thousands of distinct characters, so a normal character based BPE with a 2000 token vocabulary could not even fit all of them. The downside is that a rare Chinese character can be split into multiple byte pieces.
- **Vocabulary sizes: 2000 and 10000.** 2000 is small: 256 bytes + ~1,750 merges, so it stays close to characters for the harder languages. 10000 is 5x bigger, enough for many whole words and common Chinese characters. The gap is big enough for differences to show, while the embedding table remains small for my small Transformer later.
- **Training data:** `balanced.txt`, the shared multilingual training sentences 

In [ ]:
def train_bpe(vocab_size):
    # an empty BPE model
    tok = Tokenizer(models.BPE())
    # normalise text to Unicode
    tok.normalizer = normalizers.NFC()
    # byte level pre-tokenizer: turns text into bytes and first splits it into rough chunks
    #(words / punctuation) so merges do not cross word boundaries
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    #trainer: keep merging the most frequent pair until we reach vocab_size
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<unk>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # all 256 bytes are always in the vocab
        show_progress=False,
    )
    tok.train([f"{DATA_DIR}/tokenizer/balanced.txt"], trainer)
    return tok

bpe_2k  = train_bpe(2000)
bpe_10k = train_bpe(10000)
print("BPE-2k  vocab size:", bpe_2k.get_vocab_size())
print("BPE-10k vocab size:", bpe_10k.get_vocab_size())

bpe_2k.save(f"{OUT_DIR}/bpe_2k.json")
bpe_10k.save(f"{OUT_DIR}/bpe_10k.json")

In [ ]:
# byte level tokens are stored in a scrambled-looking alphabet. Now i am defining a helper wrapper so all three tokenizers have the same interface and tokens are printed in readable form (here `▁` marks a space).
def make_byte_decoder():
    # Inverse of the GPT-2 style byte->unicode table used by byte-level BPE.
    # It lets us turn a stored token string back into the real bytes / text.
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {chr(c): b for b, c in zip(bs, cs)}

BYTE_DECODER = make_byte_decoder()

def readable(token_str):
    # Convert a stored BPE token into readable text
    try:
        raw = bytes(BYTE_DECODER[ch] for ch in token_str)
    except KeyError:                      # special tokens such as <unk>
        return token_str
    return raw.decode("utf-8", errors="replace").replace(" ", "▁")

class BPEWrapper:
    # Gives the HF tokenizer the same methods as CharTokenizer
    def __init__(self, tok):
        self.tok = tok
    def encode_batch(self, sentences):
        return [e.ids for e in self.tok.encode_batch(sentences)]
    def to_tokens(self, text):
        return [readable(t) for t in self.tok.encode(text).tokens]
    def vocab_size(self):
        return self.tok.get_vocab_size()
    def unk_id(self):
        return self.tok.token_to_id("<unk>")
    def id_to_readable(self, i):
        return readable(self.tok.id_to_token(i))

tokenizers = {
    "char":    char_tok,
    "bpe_2k":  BPEWrapper(bpe_2k),
    "bpe_10k": BPEWrapper(bpe_10k),
}

## 1.4 Statistics on the validation set

For every tokenizer and language:
- **vocab size**: size of vocabulary
- **total tokens**: how many tokens are needed to encode the whole validation file
- **avg tokens / sentence**: total tokens divided by number of sentences
- **avg chars / token**: total unicode characters divided by total tokens

In [ ]:
rows = []
for tok_name, tok in tokenizers.items():
    for lang in LANGS:
        sents = valid[lang]
        ids = tok.encode_batch(sents)                      # list of id lists, one per sentence
        total_tokens = sum(len(x) for x in ids)            # all tokens in the validation file
        total_chars  = sum(len(s) for s in sents)          # all unicode characters in the file
        used = set(i for x in ids for i in x)              # which vocabulary entries were used
        n_unk = sum(1 for x in ids for i in x if i == tok.unk_id())
        rows.append({
            "tokenizer": tok_name,
            "language": lang,
            "vocab_size": tok.vocab_size(),
            "total_tokens": total_tokens,
            "avg_tokens_per_sentence": round(total_tokens / len(sents), 2),
            "avg_chars_per_token": round(total_chars / total_tokens, 3),
            "distinct_tokens_used": len(used),
            "unk_%": round(100 * n_unk / total_tokens, 4),
        })

results = pd.DataFrame(rows)
results.to_csv(f"{OUT_DIR}/tokenizer_stats_valid.csv", index=False)
results

In [ ]:
# a few more into it
print("Total tokens on validation set")
display(results.pivot(index="tokenizer", columns="language", values="total_tokens").loc[["char", "bpe_2k", "bpe_10k"]])
print("Average characters per token")
display(results.pivot(index="tokenizer", columns="language", values="avg_chars_per_token").loc[["char", "bpe_2k", "bpe_10k"]])

## 1.5 Analysis, Comparison and Some Examples

In [ ]:
  import re
  rng = random.Random(SEED)
  chosen = {}
  for lang in LANGS:
      pool = [s for s in valid[lang] if 30 <= len(s) <= 70 and not s.startswith(("'", "，")) and not s.endswith("「")]
      if lang == "zh":
          pool = [s for s in pool if not re.search(r"[A-Za-z]", s)]
      chosen[lang] = rng.choice(pool)
  for lang, s in chosen.items():
      print("=" * 80, f"\n[{lang}] {s}")
      for name, tok in tokenizers.items():
          p = tok.to_tokens(s)
          print(f"  {name:8s} ({len(p):3d}): " + " | ".join(p))

In [ ]:
# Report Table
report = results[["tokenizer", "language", "vocab_size", "total_tokens",
                  "avg_tokens_per_sentence", "avg_chars_per_token"]].copy()

# char -> small BPE -> large BPE
order = {"char": 0, "bpe_2k": 1, "bpe_10k": 2}
report["_o"] = report["tokenizer"].map(order)
report = report.sort_values(["_o", "language"]).drop(columns="_o")

report.columns = ["Tokenizer", "Language", "Vocab size", "Total tokens (valid)",
                  "Avg tokens / sentence", "Avg chars / token"]

report.to_csv(f"{OUT_DIR}/report_table.csv", index=False)
report.style.hide(axis="index").format({"Total tokens (valid)": "{:,}"})

In [ ]:
import matplotlib.pyplot as plt

# Same data as before: rows = tokenizers, columns = languages
order = ["char", "bpe_2k", "bpe_10k"]
pivot = results.pivot(index="tokenizer", columns="language", values="avg_chars_per_token").loc[order]

ax = pivot.plot(kind="bar", figsize=(7, 4.5))

# Write the value on top of every bar so the reader does not have to guess from the axis
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)

# Dashed line at 1.0 = "one character per token". Bars below it mean a token is LESS than one character
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1)

ax.set_ylabel("Avg Unicode characters per token")
ax.set_xlabel("Tokenizer")
ax.set_title("Compression by tokenizer and language (validation set)")
plt.xticks(rotation=0)
plt.legend(title="Language")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chars_per_token1.png", dpi=150)   
plt.show()

**Character level:** It gives exactly 1.0 characters per token, so total tokens equal the number of characters. All three languages need about 368k tokens, which is what the character balanced data leads us to expect. The vocabulary is large (9,443) because Chinese needs thousands of different characters. The average tokens per sentence differ (81, 99, 39) only because sentence lengths differ.

**BPE-2k:** English and Turkish shrink to less than half the size (about 164k and 169k tokens, roughly 2.2 characters per token). Chinese is the surprise. It needs 410k tokens, which is more than the 368k characters. Its 0.898 characters per token means each Chinese character costs on average about 1.1 tokens.

**BPE-10k:** Everything shrinks again. English and Turkish reach about 3.2 characters per token, so they are now almost identical (114.7k vs 114.3k tokens). Chinese is still far behind at 1.25 characters per token.

**Why I think the results look like this?**

- **Chinese is expensive because of bytes.** I built byte-level BPE, and one Chinese character is 3 bytes. A character only becomes one token if BPE happened to merge its 3 bytes.
  - In the 2k vocabulary there are only about 1,750 merges shared among three languages.
  - Only the very frequent Chinese characters get their own token, and the rest stay in 2–3 pieces.
  - That is why BPE-2k is worse than the plain character tokenizer for Chinese (410k vs 368k).
  - At 10k, more characters get merged, but Chinese still has thousands of different characters and a flatter frequency distribution, so it stays behind.

- **English and Turkish compress well.**
  - Letters are 1 byte each (Turkish has a few 2-byte letters).
  - Both languages repeat the same letter sequences over and over.
  - The pairs are frequent, so BPE merges them early.

## 1.6 Some more example sets

In [ ]:
test_words = {
    "en": ["the", "international", "unbelievably", "Wikipedia"],
    "tr": ["evlerimizden", "Türkiye'nin", "öğretmenlerimizin", "kitaplar"],
    "zh": ["我们", "中华人民共和国", "北京大学", "经济发展"],
}

for lang, words in test_words.items():
    for w in words:
        print(f"[{lang}] {w}")
        for name, tok in tokenizers.items():
            print(f"   {name:8s}", " | ".join(tok.to_tokens(w)))

In [ ]:
#how many Chinese tokens are broken byte fragments?
for name in ["bpe_2k", "bpe_10k"]:
    tok = tokenizers[name]
    # Which vocabulary entries are incomplete pieces of a character?
    frag_ids = {i for i in range(tok.vocab_size()) if "\ufffd" in tok.id_to_readable(i)}
    for lang in LANGS:
        flat = [i for x in tok.encode_batch(valid[lang]) for i in x]
        share = 100 * sum(1 for i in flat if i in frag_ids) / len(flat)
        print(f"{name:8s} {lang}: {share:.1f}% of tokens are partial-byte fragments")

In [ ]:
#tokens per word for English and Turkish
for name, tok in tokenizers.items():
    for lang in ["en", "tr"]:
        n_words  = sum(len(s.split()) for s in valid[lang])
        n_chars  = sum(len(w) for s in valid[lang] for w in s.split())
        n_tokens = sum(len(x) for x in tok.encode_batch(valid[lang]))
        print(f"{name:8s} {lang}: {n_chars/n_words:.1f} chars/word, {n_tokens/n_words:.2f} tokens/word")

In [ ]:
#Chinese single-character vs multi-character tokens in the vocabulary
def is_cjk(ch):
    return "\u4e00" <= ch <= "\u9fff"

for name in ["bpe_2k", "bpe_10k"]:
    tok = tokenizers[name]
    texts = [tok.id_to_readable(i) for i in range(tok.vocab_size())]
    single = [t for t in texts if len(t) == 1 and is_cjk(t)]
    multi  = [t for t in texts if len(t) >= 2 and all(is_cjk(c) for c in t)]
    print(f"{name}: {len(single)} single Chinese characters, {len(multi)} multi-character Chinese tokens")
    # Lower ids = merged earlier = more frequent in training, so these are the "strongest" units
    print("   first multi-character tokens:", multi[:15])

In [ ]:
pivot_tokens = results.pivot(index="tokenizer", columns="language", values="total_tokens").loc[order]
ax = (pivot_tokens / 1000).plot(kind="bar", figsize=(7, 4.5))        # divide by 1000 -> thousands of tokens
for container in ax.containers:
    ax.bar_label(container, fmt="%.0fk", fontsize=8, padding=2)
ax.set_ylabel("Total tokens on validation set (thousands)")
ax.set_xlabel("Tokenizer")
ax.set_title("Tokens needed to encode the validation set")
plt.xticks(rotation=0)
plt.legend(title="Language")
plt.tight_layout()
plt.savefig("outputs_part1/total_tokens.png", dpi=150)
plt.show()

## Part 2

In [19]:
GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import json, math, time, random, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "No GPU found! Check GPU_ID / nvidia-smi."
DEVICE = "cuda"
print("Using GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

# Paths
PART1_DIR = "outputs_part1"      # where Part 1 saved bpe_2k.json and bpe_10k.json
OUT_DIR   = "outputs_part2_prelim"      # where we save checkpoints, logs, tables
os.makedirs(OUT_DIR, exist_ok=True)

# Model hyperparameters
BLOCK   = 256     # context length in tokens
D_MODEL = 256     # hidden size
N_HEAD  = 4       # attention heads (256 / 4 = 64 dims per head)
N_LAYER = 2       # Transformer layers
D_FF    = 1024    # feed-forward inner size
DROPOUT = 0.1     # regularisation


# Training hyperparameters 
BATCH        = 64
LR           = 1e-3
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01
EPOCHS       = 2
EVALS_PER_EPOCH = 2
SEED         = 2326
TIE_WEIGHTS  = False

# bf16 makes training faster on modern GPUs; fall back to full precision if unsupported
# USE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
# def amp():
#     # returns a fresh "mixed precision" context (or a do-nothing context)
#     if USE_BF16:
#         return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
#     return contextlib.nullcontext()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Save the config so the run is reproducible and documented
config = dict(BLOCK=BLOCK, D_MODEL=D_MODEL, N_HEAD=N_HEAD, N_LAYER=N_LAYER, D_FF=D_FF, DROPOUT=DROPOUT,
              BATCH=BATCH, LR=LR, WARMUP_STEPS=WARMUP_STEPS, WEIGHT_DECAY=WEIGHT_DECAY,
              EPOCHS=EPOCHS, SEED=SEED, TIE_WEIGHTS=TIE_WEIGHTS)
with open(f"{OUT_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)
print(config)

Using GPU: NVIDIA GeForce GTX 1080 Ti | torch 2.4.1+cu121


NameError: name 'USE_BF16' is not defined

In [ ]:
# The seed is fixed, so every tokenizer gets exactly the same text in exactly the same order.
train_sents = [s for lang in LANGS for s in train[lang]]
random.Random(SEED).shuffle(train_sents)
valid_all = [s for lang in LANGS for s in valid[lang]]      # all validation sentences together


In [ ]:
def make_stream(tok, sentences):
    eos_id = tok.vocab_size()
    ids = []
    for sent_ids in tok.encode_batch(sentences):# encodes all sentences with this tokenizer
        ids.extend(sent_ids)# add the token ids of the sentence
        ids.append(eos_id)
    return torch.tensor(ids, dtype=torch.long, device=DEVICE)

In [ ]:
data = {}
for name, tok in tokenizers.items():
    d = {}
    d["vocab"] = tok.vocab_size() + 1# +1 because of the EOS id
    d["train"] = make_stream(tok, train_sents)
    d["valid"] = make_stream(tok, valid_all)
    d["valid_chars"] = sum(len(s) + 1 for s in valid_all)# number of characters (+1 per EOS)
    d["valid_lang"] = {}# one validation stream per language
    for lang in LANGS:
        d["valid_lang"][lang] = (make_stream(tok, valid[lang]),
                                 sum(len(s) + 1 for s in valid[lang]))
    data[name] = d